# Preprocessing Stage
### Processing 2020 Taxi data 

__This will include the following process:__

1- Reading the Taxi data stored in AWS S3 bucket.

2- Analyzing the data and cleaning the data.

3- Grouping the data in Hourly basis and Make it ready for Modeling

4- Saving the transformed data in S3 bucket



===============================================================================================




# 1- Loading the data

In [ ]:
# starting pyspark session

In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder.appName('cleaning').getOrCreate()

* __2020 taxi consist of 12 parquet files, each file contain a month taxi records data. The files are uploaded in `taxi-project-thesis` AWS Bucket inside `taxi2020` folder. To read all the 12 parquet files together we will use `.coalesce(1)` function in pyspark__

In [ ]:
# reading 12 parquet files together
df=spark.read.parquet("s3://taxi-project-thesis/taxi2020/*").coalesce(1)

In [ ]:
df.count()

__The dataframe contains `24649092` taxi trips. Lets analyze these trips.__

In [ ]:
# checking the features schema
df.printSchema()

In [ ]:
from pyspark.sql.functions import substring, to_timestamp,hour,dayofyear,month,year,weekofyear,date_format

# 2- Analyzing  and cleaning the data
## 2.1 Cleaning incorrect trips using pickup_datetime and dropoff_datetime

__We need to check are all the trips in the dataframe belong to 2020 ?__

* To do that, we will need to create new features from ` Pick-up year`,`Drop-off year`


In [ ]:
# adding the pick-up year
df_new=df.withColumn('Pick_Year',year(df['tpep_pickup_datetime']))

In [ ]:
# adding the drop-off year
df_new=df_new.withColumn('Drop_Year',year(df_new['tpep_dropoff_datetime']))

### Cleaning the trips using the pick-up year
* We need to extract the unique pick year to see if there are trips available in the dataframe which belong to incorrect years

In [ ]:
# collecting the distinct values availble in the pick-up year
df_new.select('Pick_Year').distinct().collect()

In [ ]:
from pyspark.sql.functions import *

In [ ]:
# Saving these distinct values in list
pick_year = df_new.select(collect_set('Pick_Year').alias('Pick_Year')).first()['Pick_Year']

In [ ]:
pick_year

__We have trips which occurred in diffrent years which is not correct. lets analyze these trips__

lets see how many trips occured in each year found in the Pick-up column

In [ ]:
for i in (pick_year):
    print('year ',i,df_new.filter(df_new['Pick_Year']==i).count())

__We will remove all the trips not occured in 2020. Its worth noting that there are some trips started in 2019 and ended in 2020. to understand this logic, there might be some trips picked in the last hours of 31-Dec-2019 and droped in first hour of january 2020. So we will keep 2019 trips and 2020 trips for the mean time and delete all the rest__

In [ ]:
# we will filter out the year 2020 and 2019
year_list=[2019,2020]

In [ ]:
#filtering only 2019,2020 trips data
df_new=df_new.filter(df_new.Pick_Year.isin(year_list))

In [ ]:
df.count()-df_new.count()

__There are 149 trips which have been deleted__

In [ ]:
# now lets confirm the pick up year in our dataframe
df_new.select('Pick_Year').distinct().collect()

### Cleaning the trips using the drop-off year

The assumption that all the trips should be dropped in 2020 and 2021 is similar to the one mentioned above. Since there are some trips that might be picked during the last hour of the year, they should be dropped in the first hour of 2021.

* We need to extract the unique drop year to see if there are incorrect years

In [ ]:
# checking the distinct values in the drop year column
df_new.select('Drop_Year').distinct().collect()

__So we have observations which been dropped in 2019. This is incorrect__ only trips which been dropped in 2020 and 2021 should be kept. 

In [ ]:
# we will remove the trips which been dropped in 2019
df_new.filter(df_new['Drop_Year']==2019).count()

__Now lets check the trips dropped in 2021, is our assumption is correct?__
* To do that we will create pick-up month column from `tpep_pickup_datetime` and create Drop-off month column from `tpep_dropoff_datetime`

In [ ]:
# checking how many trips have been dropped in 2021
df_new.filter((df_new['Pick_Year']==2020) & (df_new['Drop_Year']==2021)).count()

So there are trips happened in Dec 2020 and dropped in start of Jan 2021

In [ ]:
# adding pick up month
df_new=df_new.withColumn('Pick_month',month(df_new['tpep_pickup_datetime']))

In [ ]:
# adding drop month
df_new=df_new.withColumn('drop_month',month(df_new['tpep_dropoff_datetime']))

In [ ]:
df_new.filter((df_new['Pick_Year']==2020) & (df_new['Drop_Year']==2021)).select(['Pick_month','drop_month']).show(n=20)

__SO our assumption is correct, we will keep these trips.__

Now we will remove the incorrect trips which dropped in 2019

In [ ]:
year_list=[2020,2021]

In [ ]:
df_new=df_new.filter(df_new.Drop_Year.isin(year_list))

In [ ]:
#check the drop year after filtering
df_new.select('Drop_Year').distinct().collect()

In [ ]:
df_new.filter(df_new['Pick_Year']==2019).count()

63 trips have been removed. these trips are the trips dropped in 2019

__Now lets check the trips picked in 2019 and dropped in 2020__

In [ ]:
df_new.filter((df_new['Pick_Year']==2019) & (df_new['Drop_Year']==2020)).select(['Pick_month','drop_month']).show(n=100)

In [ ]:
# to make sure these 63 observations are trips picked in dec 2019 and dropped in jun 2020
df_new.filter((df_new['Pick_Year']==2019) & (df_new['Drop_Year']==2020)).select(['Pick_month','drop_month']).count()

## 2.2 Analyzing and cleaning the trip distance


Checking the trips that have zore trip distance

In [ ]:
df_new.filter(df_new['trip_distance']==0).count()

__We have `330082` trips with zero trip distance which is 0.13 % of our data__

lets analyze these trips further. we will have a look on `passenger_count` ,`trip_distance`,`fare_amount` to see if its correct or not

In [ ]:
df_new.filter(df_new['trip_distance']==0).select(['tpep_pickup_datetime',
                                                  'tpep_dropoff_datetime','passenger_count','trip_distance',
                                                  'fare_amount']).show(200)

__These are the zero distance trips,  as we can see its paid trips and there are passengers. how to find the incorrect trips ?__


* My assumption is if the drop location ID is silimar to pick location ID, then we assume there was congustion and the taxi didnt move that much or it was very short trip. so these trips are normal. On the other hand, if the drop location is different than the pick-up location, then its impossible that these trips can have zero distance and these trips are incorrect.

Lets create dataframe that only have zero distance trips

In [ ]:
zero_distance=df_new.filter(df_new['trip_distance']==0)

In [ ]:
print('Number of trips with zero trip distance ',zero_distance.count(),' trips')

In [ ]:
# trips which have zero distance and pick location is similir to drop location
zero_distance.filter(zero_distance['PULocationID']==zero_distance['DOLocationID']).count()

__So we have 157730 trips in which the pick location is similir to drop location. We will  remove the trips which have zero trip distance and the pick location is different than drop location__

In [ ]:
# trips which have zero distance and pick location is different than drop location
zero_distance.filter(zero_distance['PULocationID']!=zero_distance['DOLocationID']).count()

In [ ]:
df_new.filter((df_new['trip_distance']==0) & (df_new['PULocationID']!=df_new['DOLocationID'])).count()

In [ ]:
# our dataset should have this count
df_new.count() - 172352

### Steps to remove these `172352` trips:

* We will create data having zero distance trips (already done above).

* We will create dataframe that dont have zero distance trips.(`no_zero_distance`)


* We will filter out the trips that have zero distance and the pick up location is different than the drop locations from the zero dataframe.

* Then we will merge the zero dataframe with the non-zero-distance dataframe

In [ ]:
# 1- we will remove the trips contains zero distance
no_zero_distance=df_new.filter(df_new['trip_distance']!=0)

In [ ]:
no_zero_distance.count()

In [ ]:
#check there is no zero distance trips
no_zero_distance.filter('trip_distance=0').count()

In [ ]:
# we will create data having zero which we already created before
zero_distance.count()

In [ ]:
#checking the count of the trips with zero distance and pick location is different than drop location
zero_distance.filter(zero_distance['PULocationID']!=zero_distance['DOLocationID']).count()

In [ ]:
#3 - we will remove the trips that have zero distance and the pick up location is different than the drop locations
# here we will filter only the trips with similar pick and drop location
Zero_distance_cleared=zero_distance.filter(zero_distance['PULocationID']==zero_distance['DOLocationID'])

In [ ]:
Zero_distance_cleared.count()

In [ ]:
# to varify the correct number
no_zero_distance.count() +Zero_distance_cleared.count()

In [ ]:
# we will merge the trips that have no zero distance with the trips that have zero distance and simlar pick and drop locations
df_new_1 = no_zero_distance.union(Zero_distance_cleared)

In [ ]:
# this dataset contain all the observation except with zero distance and drop location is different than pick location
df_new_1.count()

In [ ]:
#Check if we have trips with zero distance and diffrent drop and pick locations
df_new_1.filter((df_new_1['trip_distance']==0) & (df_new_1['PULocationID']!=df_new_1['DOLocationID'])).count()

__Now we cleaned the trips that have zero distance and pick-up location is different than the drop-off location__

Lets move on to check the statistical summary of `trip_distance` column

In [ ]:
df_new_1.select('trip_distance').describe().show()

We have negative trip distance which is incorrect, lets analyze the count and remove this trips.

In [ ]:
df_new_1.filter(df_new_1['trip_distance']<0).count()

In [ ]:
# We will remove these trips
df_new_1=df_new_1.filter(df_new_1['trip_distance']>=0)

__Checking the outliers__

In [ ]:
 # Q1 : First Quartile ., Q3 : Third Quartile
Q1 = df_new_1.approxQuantile('trip_distance',[0.25],relativeError=0.05)
Q3 = df_new_1.approxQuantile('trip_distance',[0.75],relativeError=0.05)

In [ ]:
print('Q1: ',Q1, 'Q3: ',Q3)

In [ ]:
df_new_1.count()

In [ ]:
df_new_1.filter(df_new_1['trip_distance']>2.8).count()

__We have 75% of our trips have trip distance below 2.8 mile__

In [ ]:
#IQR : Inter Quantile Range
#We need to define the index [0], as Q1 & Q3 are a set of lists
IQR = Q3[0] - Q1[0]

In [ ]:
#selecting the data, with -1.5*IQR to + 1.5*IQR., where param = 1.5 default value
less_Q1 =  Q1[0] - 1.5*IQR
more_Q3 =  Q3[0] + 1.5*IQR

In [ ]:
more_Q3

In [ ]:
df_new_1.filter(df_new_1['trip_distance']>more_Q3).count()

__Statistically, any trip have more than 5.5 mile trip distance is an outlier. We have around 12% of our data contains trips above 5.5 mile. But in reality trips more than 5.5 doesnt sound like incorrect. and outliers will not affect our model since we are going to aggregate the counts of trips. so we will just remove the unrealistic values__

In [ ]:
df_new_1.groupBy('trip_distance').count().orderBy(df_new_1['trip_distance'].desc()).show(50)

These all looks like unrealistic trips, lets check the percentile values

In [ ]:
print('The 90th percentile of trip distance is: ',df_new_1.approxQuantile('trip_distance',[0.90],relativeError=0.05)[0])

In [ ]:
print('The 94th percentile of trip distance is: ',df_new_1.approxQuantile('trip_distance',[0.94],relativeError=0.05)[0])

In [ ]:
df_new_1.filter(df_new_1['trip_distance']>8.76).count()

Around 6 % from the trips is above 8.76 miles . but still 8.76 mile per trip is realistic. we will just try to remove the unrealistic trips by removing `any trip above 1000 mile`

In [ ]:
# Number of trips above or equal 1000 miles
df_new_1.filter(df_new_1['trip_distance']>=1000).count()

In [ ]:
df_new_1=df_new_1.filter(df_new_1['trip_distance']<1000)

In [ ]:
df_new_1.count()

## 2.3 Fare amount
As per https://www1.nyc.gov/site/tlc/passengers/taxi-fare.page the min fare amount is $2.50 initial charge. so we will analyze the fare amount

In [ ]:
# count of trips that have fare amount less than 2.5 dollars 
df_new_1.filter(df_new_1['fare_amount']<2.5).count()

In [ ]:
#removing the trips that have fare amount less than 2.5
df_new_1=df_new_1.filter(df_new_1['fare_amount']>=2.5)

In [ ]:
df_new_1.select('fare_amount').describe().show()

As we can see, the average fare amount is 12.6 $ but we have unrealistic fare_amount which need to be removed.

In [ ]:
df_new_1.groupBy('fare_amount').count().orderBy(df_new_1['fare_amount'].desc()).show(50)

__Checking the outliers__

In [ ]:
 # Q1 : First Quartile ., Q3 : Third Quartile
Q1 = df_new_1.approxQuantile('fare_amount',[0.25],relativeError=0.05)
Q3 = df_new_1.approxQuantile('fare_amount',[0.75],relativeError=0.05)

In [ ]:
#IQR : Inter Quantile Range
#We need to define the index [0], as Q1 & Q3 are a set of lists
IQR = Q3[0] - Q1[0]

In [ ]:
#selecting the data, with -1.5*IQR to + 1.5*IQR., where param = 1.5 default value
less_Q1 =  Q1[0] - 1.5*IQR
more_Q3 =  Q3[0] + 1.5*IQR

In [ ]:
more_Q3

In [ ]:
df_new_1.filter(df_new_1['fare_amount']>more_Q3).count()

We have 9.6% of our trips have fare amount 22.75 dollars and considered as outliers. but still we will keep them and remove the unrealistic values

In [ ]:
df_new_1.filter(df_new_1['fare_amount']>500).count()

__we will remove the trips `above 500 dollars` since its not realistic in my point of view and only 113 trips which will not affect our data__

In [ ]:
df_new_1=df_new_1.filter(df_new_1['fare_amount']<=500)

In [ ]:
df_new_1.count()

## 2.4 Passenger

The number of passengers in the vehicle.
This is a driver-entered value.

In [ ]:
df_new_1.select('passenger_count').describe().show()

In [ ]:
df_new_1.filter(df_new_1['passenger_count']==0).count()

Since its the taxi driver entry, maybe its was written by mistake. so will not remove the trips with zero passenger count

## 2.5 Trip duration
As per NYC Taxi & Limousine Commision Regulations, the maximum duration taxi driver can drive per day is 12 hours, so we will assume that the max trip duration could be 12 hours and remove and records axceed 12 hours

Steps to remove the trips above 12 hours:

* We will create new column that contain the trip duration by second `trip_duration_seconds` (just subtract the UNIX pickup time from the UNIX drop time)

* We will create trip_duration in hours from the newly created `trip_duration_seconds`

* Then filterout and remove any trip above 12 hours

In [ ]:
# Create new coloumn trip_duration_seconds
df_new_1.withColumn('trip_duration_seconds',unix_timestamp("tpep_dropoff_datetime") - unix_timestamp('tpep_pickup_datetime'))\
.select(['tpep_pickup_datetime','tpep_dropoff_datetime','trip_duration_seconds']).show(20)

In [ ]:
df_new_1=df_new_1.withColumn('trip_duration_seconds',unix_timestamp("tpep_dropoff_datetime") - unix_timestamp('tpep_pickup_datetime'))

In [ ]:
#We will create trip duration in hour which is the `trip_duration_seconds / 3600`
df_new_1=df_new_1.withColumn('duration_In_Hours',round(col('trip_duration_seconds')/3600,1))

In [ ]:
df_new_1.filter(df_new_1['duration_In_Hours']>12).count()

In [ ]:
df_new_1=df_new_1.filter(df_new_1['duration_In_Hours']<=12)

In [ ]:
df_new_1.filter(df_new_1['duration_In_Hours']>12).count()

## 2.6 Check the trips in each month

The trips in each month shpuld be dropped in the same month and some trips in the next month. for example january trips will have pick-up trips occurred in january and the drop of these trips must be in january and some of them will be dropped in february, following the assumption that some trips have been picked during the last hour in 31-Jan and it might been dropped in the first hour during the february first.

In [ ]:
for i in range(1,13):
    print('Pick up month ',i,df_new_1.filter(df_new_1['Pick_month']==i).select('drop_month').distinct().collect())

__We notice that in March trips we have trips dropped in feb which is incorrect. lets view this data__

In [ ]:
df_new_1.filter((df_new_1['Pick_month']==3) & (df_new_1['drop_month']==2)).count()

In [ ]:
df_new_1.filter((df_new_1['Pick_month']==3) & (df_new_1['drop_month']==2)).select(['tpep_pickup_datetime','tpep_dropoff_datetime']).show()

its wrong trip which should be removed

__Steps to remove this trip:__

* create dataset without march

* create dataframe only for March trips

* filter out this row and union the two dataframe

In [ ]:
df_new_1.count()

In [ ]:
df_new_1.filter(df_new_1['Pick_month']==3).count()

In [ ]:
df_new_1.filter(df_new_1['Pick_month']!=3).count()

In [ ]:
# create march_df
march=df_new_1.filter(df_new_1['Pick_month']==3)

In [ ]:
# create no_march_df
no_march_df = df_new_1.filter(df_new_1['Pick_month']!=3)

In [ ]:
# filter out the feb drop row
march.filter(march['drop_month']==2).count()

In [ ]:
march=march.filter(march['drop_month']!=2)

In [ ]:
march.select('drop_month').distinct().collect()

In [ ]:
df_new_final = march.union(no_march_df)

In [ ]:
df_new_final.count()

In [ ]:
df_new_1.count() - df_new_final.count()

In [ ]:
# check the drop year for the trips happened in Dec 2020
df_new_final.filter(df_new_final['Pick_month']==12).select('Drop_Year').distinct().collect()

__Everything look cleaned now__

we will move to group and aggregate the trips data

# 3- Grouping the data per location ID in Hourly basis and Make it ready for Modeling

__We will need to groupby and aggregate the count of the taxi trips per location ID  to be in hourly basis__

Lets have look how the data look like

In [ ]:
df_new_final.select(['tpep_pickup_datetime','PULocationID']).show()

__Lets check how many Location ID available in our dataframe__

In [ ]:
df_new_final.select(countDistinct("PULocationID")).show()

__We have 262 location ID, so when we will aggregate the the taxi trips in hourly basis, we should have `366 X 24 X 262 = 2301408 rows`__

- 366 days in 2020

- 24 hours per day

- 262 location ID

### Steps:
We will change the timestamp to this format `"yyyy-MM-dd HH:00"` to be able to group the trips hourly

In [ ]:
df_new_final=df_new_final.withColumn("Pickup_datetime_hourly", date_format(col("tpep_pickup_datetime").cast("timestamp"), "yyyy-MM-dd HH:00"))

In [ ]:
df_new_final.select(['tpep_pickup_datetime','Pickup_datetime_hourly','PULocationID']).show()

__we will create trip count column__

In [ ]:
# we will create trip count column
df_new_final=df_new_final.withColumn("Trip_count", lit(1))

### Now lets groupby and aggregate the trips per location in hourly basis

In [ ]:
hourly_aggregated=df_new_final.groupby(['Pickup_datetime_hourly','PULocationID']).agg({'Trip_count':'count'})\
.withColumnRenamed("count(Trip_count)", "Trips_count")

In [ ]:
hourly_aggregated.show()

In [ ]:
hourly_aggregated.count()

__The aggregated trips have `910784` rows and it should be `2301408`, and I think the issue is there are no pick-up records in certain hours. So when I gourpby and aggregate the data, it will not show the missing hours. 

#### To solve this problem:
- we need to create dataframe with timestamp coloumn started from 1-1-2020 till 1-1-2021. `(8784 rows)`.
- We will use the timestamp dataframe which we create in the first step and use it to generate dataframe that contains timestamp column and another coloumn per location ID and then merge all of them. in this case we will have `(2301408)` rows and two columns, timestamp and LocationID.
- Create unique column that contain the __timestamp plus the locationID__ in both `hourly_aggregated` and ` generated timestamp dataframe` (the dataframe we created in the previous step) so we can join them.
- Then left join the generated timestamp dataframe with our hourly aggregated dataframe.

### 1- Creating unique coloum in hourly_aggregated
The unique column will merge the timestamp with the locationID. we need to convert the locationID from integer to string then we will use `concat_ws` to merge the string values together

In [ ]:
hourly_aggregated.printSchema()

In [ ]:
from pyspark.sql.types import TimestampType,StructField,StringType,IntegerType,StructType

In [ ]:
#Create new column which convert the location ID to string
hourly_aggregated= hourly_aggregated.withColumn("PULocationID_string", hourly_aggregated["PULocationID"].cast(StringType()))

In [ ]:
#Create the unique column using concat_ws
hourly_aggregated=hourly_aggregated.select(concat_ws('_',hourly_aggregated.Pickup_datetime_hourly,
                                   hourly_aggregated.PULocationID_string).alias("UniqueColumn"),"Pickup_datetime_hourly","PULocationID","Trips_count")

In [ ]:
hourly_aggregated.show(5)

In [ ]:
hourly_aggregated.printSchema()

### 2- Generate timestamp dataframe from 1-1-2020 till 1-1-2021
The below function will generate spark timestamp dataframe 

In [ ]:
def generate_series(start, stop, interval):
    """
    :param start  - lower bound, inclusive
    :param stop   - upper bound, exclusive
    :interval int - increment interval in seconds
    """
    start, stop = spark.createDataFrame(
        [(start, stop)], ("start", "stop")
    ).select(
        [col(c).cast("timestamp").cast("long") for c in ("start", "stop")
    ]).first()
    # Create range with increments and cast to timestamp
    return spark.range(start, stop, interval).select(
        col("id").cast("timestamp").alias("timestamp")
    )

In [ ]:
# we will multiply by 60*60 to change the timestamp to hour
timestamp_df=generate_series("2020-01-01", "2021-01-01", 60 * 60)

__We will change the generated timestamp to the same format in `hourly_aggregated`__

In [ ]:
timestamp_df=timestamp_df.withColumn("hourly_timestamp", date_format(col("timestamp").cast("timestamp"), "yyyy-MM-dd HH:00"))

In [ ]:
timestamp_df.printSchema()

In [ ]:
timestamp_df.count()

__Now this dataframe contain hourly timestamp for 2020. so we need to create the same rows for each location ID exist in `hourly_aggregated`__

Lets extract the location ID from the hourly_aggregated and save them into list

In [ ]:
# extract the pickup location id from the dataset and save it in list
Pick_up_LocationID = df_new_final.select(collect_set('PULocationID').alias('locationID_list')).first()['locationID_list']

In [ ]:
len(Pick_up_LocationID)

__We have 262 location ID, we will aggregate the trips in hourly basis. so we should have `366 X 24 X 262 = 2301408` rows.__

To do so, we need to iterate generated timestamp for each locationID (for loop) and append the result to an empty spark dataframe. in this case, we will have 2 columns and 2301408 rows.

__Define function that will help us to add the location ID column to the generated timestamp__

In [ ]:
def locations_timestampGenerator(dataframe,locationID):
    dataframe=dataframe.withColumn("locationID", lit(locationID))
    return dataframe

__Now lets create an empty spark dataframe contains two columns `timestamp` and `locationID`__

In [ ]:
data_schema = [StructField("timestamp",TimestampType(), True),StructField("hourly_timestamp", StringType(), True),
               StructField("locationID", IntegerType(), True)]

In [ ]:
final_struct = StructType(fields=data_schema)

In [ ]:
ID_plus_timestamp = spark.createDataFrame([], schema=final_struct)

In [ ]:
# iterate generated timestamp for each locationID (for loop) 
for i in Pick_up_LocationID:
    dataframe=locations_timestampGenerator(timestamp_df,i)
    ID_plus_timestamp = ID_plus_timestamp.union(dataframe)

In [ ]:
ID_plus_timestamp.count()

__Now lets make sure that all the locationID created by the `for loop` is matching the existing locationID in the original aggregated dataset__

We will extract the distinct location ID from `ID_plus_timestamp` and check with the location ID in `Hourly_aggregated`

In [ ]:
# extract the distinct location ID from ID_plus_timestamp
location_ID_list2=ID_plus_timestamp.select(collect_set('locationID').alias('locationID_list')).first()['locationID_list']

__Remember we already extracted the location ID from Hourly_aggregated `Pick_up_LocationID`, so we dont need to do it again__

now if there is any location ID in `ID_plus_timestamp`dataframe not exist in `Hourly_aggregated` it will append it to the empty list

In [ ]:
unique_list = []
 
for item in location_ID_list2: 
    if item not in Pick_up_LocationID: 
        unique_list.append(item)

In [ ]:
# Should be empty
unique_list

#### Last step, We will need to create unique column that contain the __timestamp plus the locationID__ in `ID_plus_timestamp` so we can join it to `hourly_aggregated`.

In [ ]:
# Change the location ID from integer to string
ID_plus_timestamp= ID_plus_timestamp.withColumn("locationID_string", ID_plus_timestamp["locationID"].cast(StringType()))

In [ ]:
# Create the unique column using concat_ws
ID_plus_timestamp=ID_plus_timestamp.select(concat_ws('_',ID_plus_timestamp.hourly_timestamp,
                                   ID_plus_timestamp.locationID_string).alias("UniqueColumn_G"),"timestamp","hourly_timestamp","locationID")

In [ ]:
ID_plus_timestamp.show(3)

#### Now joining both dataframes using the `unique column`

In [ ]:
# join both dataframes using the unique column
Pick_aggregated=hourly_aggregated.join(ID_plus_timestamp,hourly_aggregated.UniqueColumn ==  ID_plus_timestamp.UniqueColumn_G,"right")

__Now after we joined the dataframes, we will need to replace null values in PULocationID and count(Trip_count) columns of dataframe hourly_aggregated_final with 0.__

Assume  PULocationID of missing data to be 0

In [ ]:
Pick_aggregated=Pick_aggregated.na.fill(0,subset=["PULocationID"])

In [ ]:
Pick_aggregated=Pick_aggregated.na.fill(0,subset=["Trips_count"])

#### Now lets select the columns we need : `timestamp`, `locationID` ,` Trips_count` and the `UniqueColumn_G`
I have selected `UniqueColumn_G` to be the unique identifier and to join it with the dropped aggregated dataframe

In [ ]:
Pick_aggregated=Pick_aggregated.select(['timestamp','hourly_timestamp','locationID','Trips_count','UniqueColumn_G'])

In [ ]:
Pick_aggregated.show()

## Aggregating the drop trips per location ID in hourly basis

- We will follow the same steps how we aggregate the pick-up trips above, but now for the drop-off so we will have hourly count per location ID for the pick-up trips and drop-off trips

In [ ]:
df_new_final.select(countDistinct("DOLocationID")).show()

__We have 263 drop location ID, and we have 262 pick-up location ID, we will remove the drop trips which belong to the extra location ID__

In [ ]:
# extract the pickup location id from the dataset and save it in list
Drop_LocationID = df_new_final.select(collect_set('DOLocationID').alias('DOLocationID')).first()['DOLocationID']

In [ ]:
unique_list = []
 
for item in Drop_LocationID: 
    if item not in Pick_up_LocationID: 
        unique_list.append(item)

In [ ]:
unique_list

so we will remove the drop trips for location ID = 104

In [ ]:
Drop_df=df_new_final.filter(df_new_final['DOLocationID']!=104)

#### Now we are going to aggregate the data based on hourly drop count per locationID.
We will change the timestamp to this format `"yyyy-MM-dd HH:00"` to be able to group the trips hourly

In [ ]:
Drop_df=Drop_df.withColumn("Drop_datetime_hourly", date_format(col("tpep_dropoff_datetime").cast("timestamp"), "yyyy-MM-dd HH:00"))

In [ ]:
# Grouping and aggregating the dropped trips
drop_aggregated = Drop_df.groupby(['Drop_datetime_hourly','DOLocationID']).agg({'Trip_count':'count'})\
.withColumnRenamed("count(Trip_count)", "Drop_Trips_count")

In [ ]:
drop_aggregated.show()

__Create unique column that contain the timestamp plus the locationID in ID_plus_timestamp so we can join it to Pick_aggregated__

Convert the DOLocationID to string type

In [ ]:
drop_aggregated=drop_aggregated.withColumn("DOLocationID_string", drop_aggregated["DOLocationID"].cast(StringType()))

In [ ]:
# Create the unique column using `concat_ws`
drop_aggregated=drop_aggregated.select(concat_ws('_',drop_aggregated.Drop_datetime_hourly,
                                   drop_aggregated.DOLocationID_string).alias("UniqueColumn"),"Drop_datetime_hourly","Drop_datetime_hourly","DOLocationID","Drop_Trips_count")

### Joining Pick_aggregated & drop_aggregated dataframes using the UniqueColumn

In [ ]:
final_df=drop_aggregated.join(Pick_aggregated,drop_aggregated.UniqueColumn == Pick_aggregated.UniqueColumn_G,"right")

In [ ]:
# Filling the null Drop_Trips_count with zero
final_df=final_df.na.fill(0,subset=["Drop_Trips_count"])

### Now we will just select the columns we  want:

`hourly timestamp`, `locationID`, `Pick_Trips_count` and `Drop_Trips_count`

In [ ]:
final_df=final_df.select(['timestamp','hourly_timestamp','locationID','Trips_count','Drop_Trips_count'])

In [ ]:
final_df=final_df.withColumnRenamed('Trips_count','Pick_Trips_count')

In [ ]:
final_df.show()

# Feature Engineering

Now the dataframe is ready for modeling, final step is just to create some features such as month, day, year, hour and dayofweek


In [ ]:
# Create Year column
final_df=final_df.withColumn('Year',year(final_df['timestamp']))

In [ ]:
# Create Month column
final_df=final_df.withColumn('Month',month(final_df['timestamp']))

In [ ]:
# create day of month column
final_df=final_df.withColumn('DayOfMonth',dayofmonth(final_df['timestamp']))

In [ ]:
# Create hour column
final_df=final_df.withColumn('Hour',hour(final_df['timestamp']))

In [ ]:
# Create Day of week column
final_df=final_df.withColumn("dayofweek", dayofweek(col("timestamp")))

In [ ]:
final_df.show()

### Save the final dataframe as parquet file in S3

__Now we will save this spark dataframe to S3 bucket inside `EMR-project folder`__

In [ ]:
final_df.coalesce(1).write.parquet("s3://taxi-project-thesis/EMR-project/final_2020")

In [ ]:
# CHeck if we can load the final_2020 dataframe 

In [ ]:
### Load the final dataframe from S3
df_taxi=spark.read.parquet("s3://taxi-project-thesis/EMR-project/final_2020/part-00000-6b94e604-fe74-416d-a81c-d4efd46dd9cb-c000.snappy.parquet")

In [ ]:
df_taxi.show(5)

In [ ]:
df_taxi.count()

DONE !